# 03 · NEU Steel Defects - Transfer Learning + MLflow

6-class classification of steel surface defects. All runs are logged to MLflow.

In [ ]:
import sys
from pathlib import Path

_nb_root = Path('../..').resolve()
if str(_nb_root) not in sys.path:
    sys.path.insert(0, str(_nb_root))

from utils.arkon_utils import (
    get_device, get_mlflow_uri, save_figure,
    Timer, CheckpointManager, recommended_num_workers
)
print('arkon_utils loaded ✓')

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from pathlib import Path
import numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import classification_report, f1_score
import mlflow, mlflow.pytorch

device = get_device()
DEVICE = device
ASSETS = 'cv/neu'


In [ ]:
import mlflow
import mlflow.pytorch

# MLflow - platform-safe SQLite (works on Windows + Mac)
mlflow.set_tracking_uri(get_mlflow_uri())
EXPERIMENT_NAME = "arkon-cv-neu"
RUN_NAME          = "resnet18_v1"

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT_NAME)
print(f'MLflow URI: {mlflow.get_tracking_uri()}')

# Tags describe the model context - visible in the MLflow UI
MLFLOW_TAGS = {"dataset": "neu_det", "task": "multiclass_classification", "framework": "pytorch", "architecture": "resnet18", "num_classes": "6", "department": "Arkon_SteelRolling"}
print(f"MLflow tracking URI : {MLFLOW_URI}")
print(f"Experiment          : {EXPERIMENT_NAME}")
print(f"Open UI at          : {MLFLOW_URI}")

In [ ]:
DATA_DIR  = Path('../../../data/04_neu/raw')
MODEL_DIR = Path('../../../models/04_neu')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

PARAMS = {
    "img_size"      : 224,
    "batch_size"    : 64 if device.type == "cuda" else 32,
    "epochs"        : 20,
    "lr"            : 1e-4,
    "dropout"       : 0.4,
    "model_arch"    : "resnet18",
    "num_classes"   : 6,
    "num_workers"   : recommended_num_workers(device),
    "pin_memory"    : device.type == "cuda",
    "seed"          : 42,
}
torch.manual_seed(PARAMS["seed"])
if torch.cuda.is_available(): torch.cuda.manual_seed(PARAMS["seed"])
print(f'Config: {PARAMS}')


## 1. DataLoaders

In [ ]:
MEAN = [0.485, 0.456, 0.406]; STD = [0.229, 0.224, 0.225]; SZ = PARAMS["img_size"]
train_tf = transforms.Compose([
    transforms.Resize((SZ, SZ)), transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
val_tf = transforms.Compose([
    transforms.Resize((SZ, SZ)), transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

train_set = datasets.ImageFolder(DATA_DIR/'train'/'images',      transform=train_tf)
val_set   = datasets.ImageFolder(DATA_DIR/'validation'/'images', transform=val_tf)
CLASSES   = train_set.classes
BS        = PARAMS["batch_size"]
train_loader = DataLoader(train_set, BS, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_set,   BS, shuffle=False, num_workers=0)
print(f"Classes: {CLASSES}")
print(f"Train: {len(train_set)} | Val: {len(val_set)}")

## 2. Model

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Sequential(nn.Dropout(PARAMS["dropout"]),
                         nn.Linear(512, PARAMS["num_classes"]))
model = model.to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=PARAMS["lr"])
scheduler = CosineAnnealingLR(optimizer, T_max=PARAMS["epochs"])
print(f"Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} params")

## 3. Training + MLflow

In [ ]:
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

def run_epoch(model, loader, criterion, optimizer, device, train=True):
    model.train(train)
    loss_sum, correct, total = 0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            with torch.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                out  = model(X)
                loss = criterion(out, y)
            if train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            loss_sum += loss.item() * len(y)
            correct  += (out.argmax(1) == y).sum().item()
            total    += len(y)
    return loss_sum / total, correct / total


In [ ]:
CKPT_DIR = Path('../../../models/checkpoints/neu_resnet18')
ckpt = CheckpointManager(CKPT_DIR)

if ckpt.exists_torch('neu_resnet18_best'):
    print('\n⚡ Checkpoint found - skipping training')
    state, meta = ckpt.load_torch('neu_resnet18_best', map_location=device)
    model.load_state_dict(state)
    history = meta.get('history', {})
    train_time_str = meta.get('train_time', 'unknown')
    print(f'  best_epoch   : {meta.get("best_epoch", "?")}')
    print(f'  val_acc      : {meta.get("val_acc", meta.get("val_f1", "?"))}')
    print(f'  train time   : {train_time_str}')
else:
    with Timer('neu_resnet18 training') as train_timer:
        with mlflow.start_run(run_name=RUN_NAME, tags=MLFLOW_TAGS) as run:
            mlflow.log_params(PARAMS)
            print(f"Run ID: {run.info.run_id}")
        
            history = {k: [] for k in ['train_loss','val_loss','train_acc','val_acc']}
            best_val_acc = 0.0
        
            print(f"{'Ep':>3} | {'TrLoss':>8} | {'TrAcc':>7} | {'VlLoss':>8} | {'VlAcc':>7}")
            print("-" * 48)
            for epoch in range(1, PARAMS["epochs"]+1):
                tl, ta = run_epoch(model, train_loader, criterion, optimizer, DEVICE, train=True)
                vl, va = run_epoch(model, val_loader,   criterion, None,      DEVICE, train=False)
                scheduler.step()
                mlflow.log_metrics({"train_loss": round(tl,6), "val_loss": round(vl,6),
                                    "train_acc": round(ta,6), "val_acc": round(va,6)}, step=epoch)
                for k, v in zip(['train_loss','val_loss','train_acc','val_acc'],[tl,vl,ta,va]):
                    history[k].append(v)
                if va > best_val_acc:
                    best_val_acc = va
                    torch.save(model.state_dict(), MODEL_DIR / 'resnet18_neu_best.pth')
                print(f"{epoch:>3} | {tl:>8.4f} | {ta:>7.4f} | {vl:>8.4f} | {va:>7.4f}")
        
            # ── Training curves ───────────────────────────────────────────────────────
            fig, (ax1,ax2) = plt.subplots(1,2,figsize=(12,4))
            ax1.plot(history['train_loss'],label='Train'); ax1.plot(history['val_loss'],label='Val')
            ax1.set_title('Loss'); ax1.legend()
            ax2.plot(history['train_acc'],label='Train'); ax2.plot(history['val_acc'],label='Val')
            ax2.set_title('Accuracy'); ax2.legend()
            plt.suptitle(f'NEU ResNet18 - 6 classes | best_val_acc={best_val_acc:.4f}')
            plt.tight_layout()
            with tempfile.TemporaryDirectory() as tmp:
                p = os.path.join(tmp,"training_curves.png")
                plt.savefig(p, dpi=120); save_figure(fig, 'cv_neu_plot_1', subfolder='cv/neu')
plt.show(); mlflow.log_artifact(p, "plots")
        
            # ── Confusion matrix ──────────────────────────────────────────────────────
            model.load_state_dict(torch.load(MODEL_DIR/'resnet18_neu_best.pth', map_location=DEVICE))
            model.eval()
            preds, true_labels = [], []
            with torch.no_grad():
                for X, y in val_loader:
                    preds.extend(model(X.to(DEVICE)).argmax(1).cpu().tolist())
                    true_labels.extend(y.tolist())
        
            report = classification_report(true_labels, preds, target_names=CLASSES, output_dict=True)
            mlflow.log_metrics({
                "val_accuracy"  : round(report["accuracy"], 6),
                "val_f1_macro"  : round(report["macro avg"]["f1-score"], 6),
                "val_f1_weighted": round(report["weighted avg"]["f1-score"], 6),
                "best_val_acc"  : round(best_val_acc, 6),
            })
            print(classification_report(true_labels, preds, target_names=CLASSES))
        
            cm = confusion_matrix(true_labels, preds)
            fig2, ax = plt.subplots(figsize=(8,7))
            ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(cmap='Blues', ax=ax, xticks_rotation=45)
            ax.set_title('NEU ResNet18 - Val Confusion Matrix')
            plt.tight_layout()
            with tempfile.TemporaryDirectory() as tmp:
                p = os.path.join(tmp,"confusion_matrix.png")
                plt.savefig(p, dpi=120); save_figure(fig, 'cv_neu_plot_1', subfolder='cv/neu')
plt.show(); mlflow.log_artifact(p, "plots")
        
            # ── Model registry ────────────────────────────────────────────────────────
            mlflow.pytorch.log_model(model, "model", registered_model_name="neu_resnet18")
            mlflow.log_artifact(str(MODEL_DIR/'resnet18_neu_best.pth'), "weights")
            print(f"\n✅ Run complete | best_val_acc={best_val_acc:.4f}")
    train_time_str = train_timer.report()
    # Save best checkpoint with metadata
    ckpt.save_torch(model.state_dict(), 'neu_resnet18_best',
                    metadata={'train_time': train_time_str, 'history': history})


## Summary

| Parameter | Value |
|---|---|
| MLflow Experiment | `neu_steel_defect_classification` |
| Model Registry | `neu_resnet18` |
| Key metric | val_f1_macro (6 classes) |
| Server | `http://SQLite (mlflow.db)` |

In [ ]:
# ── Training curves ──────────────────────────────────────────────
if history:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ep = range(1, len(history.get('train_loss', [])) + 1)
    axes[0].plot(ep, history['train_loss'], label='Train')
    axes[0].plot(ep, history['val_loss'],   label='Val')
    axes[0].set_title('Loss'); axes[0].legend()
    axes[1].plot(ep, history['train_acc'], label='Train')
    axes[1].plot(ep, history['val_acc'],   label='Val')
    axes[1].set_title('Accuracy'); axes[1].legend()
    plt.suptitle(f'ResNet18 - NEU 6-class  |  {train_time_str}')
    plt.tight_layout()
    save_figure(fig, 'cv_neu_training_curves', subfolder='cv/neu')
    plt.show()
